In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
import gc, time, glob

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"device: {device}, GPUs available: {n_gpus}")
if device == "cuda":
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")

def show_gpu_mem(tag=""):
    if device == "cuda":
        for i in range(n_gpus):
            alloc = torch.cuda.memory_allocated(i) / 1e9
            reserved = torch.cuda.memory_reserved(i) / 1e9
            print(f"[{tag}] GPU {i}: {alloc:.2f} GB allocated, {reserved:.2f} GB reserved")

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()

show_gpu_mem("startup")


device: cuda, GPUs available: 2
GPU 0: Tesla T4
[startup] GPU 0: 0.00 GB allocated, 0.00 GB reserved
[startup] GPU 1: 0.00 GB allocated, 0.00 GB reserved


In [ ]:
def find_samanantar_files():
    en_candidates = glob.glob("/kaggle/input/**/*.en", recursive=True)
    te_candidates = glob.glob("/kaggle/input/**/*.te", recursive=True)
    return en_candidates, te_candidates

en_candidates, te_candidates = find_samanantar_files()
print("Found .en files:", en_candidates)
print("Found .te files:", te_candidates)

eng_path = Path(en_candidates[0]) if en_candidates else Path("/kaggle/input/PATH_TO/train.en")
tel_path = Path(te_candidates[0]) if te_candidates else Path("/kaggle/input/PATH_TO/train.te")

def load_data(eng_path, tel_path):
    with open(eng_path, mode="r", encoding="utf-8") as f:
        eng = f.read().splitlines()
    with open(tel_path, mode="r", encoding="utf-8") as f:
        tel = f.read().splitlines()
    assert len(eng) == len(tel), "English and Telugu files must have the same number of lines"
    return eng, tel

train_eng, train_tel = load_data(eng_path, tel_path)
print(f"Total pairs available: {len(train_eng)}")
print("Sample:", train_eng[0], "->", train_tel[0])


Found .en files: ['/kaggle/input/datasets/kaushikrajagiri/english-to-telugu/en-te/train.en']
Found .te files: ['/kaggle/input/datasets/kaushikrajagiri/english-to-telugu/en-te/train.te']
Total pairs available: 4841862
Sample: Rise again. -> మళ్లీ ఉదయిస్తాడు.


In [ ]:
subset_size = 10000
train_eng = train_eng[:subset_size]
train_tel = train_tel[:subset_size]
print(f"Using {len(train_eng)} sentence pairs")


Using 10000 sentence pairs


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenization_in_batch(texts, tokenizer, Max_length, batch_size=512):
    all_input_ids, all_attention_masks = [], []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoding = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=Max_length,
            return_tensors="pt",
        )
        all_input_ids.append(encoding["input_ids"])
        all_attention_masks.append(encoding["attention_mask"])
    return {
        "input_ids": torch.cat(all_input_ids, dim=0),
        "attention_mask": torch.cat(all_attention_masks, dim=0),
    }

probe = tokenization_in_batch(train_tel, tokenizer, Max_length=128)
lengths = probe["attention_mask"].sum(dim=1)
print(f"Telugu token length -> mean: {lengths.float().mean():.1f}, "
      f"median: {lengths.float().median():.1f}, max: {lengths.max().item()}")
del probe


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Telugu token length -> mean: 19.5, median: 14.0, max: 128


In [ ]:
Max_length = 64

encoded_eng = tokenization_in_batch(train_eng, tokenizer, Max_length=Max_length)
encoded_tel = tokenization_in_batch(train_tel, tokenizer, Max_length=Max_length)

encoded_eng = {k: v.cpu() for k, v in encoded_eng.items()}
encoded_tel = {k: v.cpu() for k, v in encoded_tel.items()}

print(encoded_eng["input_ids"].shape, encoded_tel["input_ids"].shape)


torch.Size([10000, 64]) torch.Size([10000, 64])


In [ ]:
multi_model = AutoModel.from_pretrained("xlm-roberta-base").to(device)
multi_model.eval() 
for p in multi_model.parameters():
    p.requires_grad = False

show_gpu_mem("after loading XLM-R")


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[after loading XLM-R] GPU 0: 1.11 GB allocated, 1.17 GB reserved
[after loading XLM-R] GPU 1: 0.00 GB allocated, 0.00 GB reserved


In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)
        return output, (hidden, cell)


In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)


In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=1)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
        self.attention = attention

    def forward(self, x, hidden, cell, encoder_outputs):
        embedded = self.embedding(x)                                      
        attn_weights = self.attention(hidden.squeeze(0), encoder_outputs)  
        attn_weights = attn_weights.unsqueeze(1)                          
        context = torch.bmm(attn_weights, encoder_outputs)                
        lstm_input = torch.cat((embedded, context), dim=2)
        outputs, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        prediction = self.fc(torch.cat((outputs, context), dim=2))          
        return prediction, hidden, cell


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, use_checkpointing=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.use_checkpointing = use_checkpointing

    def forward(self, src_embedded, tel_batch):
        encoder_outputs, (hidden, cell) = self.encoder(src_embedded)
        seq_length = tel_batch.size(1)
        outputs = []
        x = tel_batch[:, 0:1]

        for t in range(1, seq_length):
            if self.use_checkpointing:
                prediction, hidden, cell = checkpoint(
                    self.decoder, x, hidden, cell, encoder_outputs, use_reentrant=False
                )
            else:
                prediction, hidden, cell = self.decoder(x, hidden, cell, encoder_outputs)
            outputs.append(prediction)
            x = tel_batch[:, t:t+1]  

        return torch.cat(outputs, dim=1)  


In [ ]:
HIDDEN_DIM = 256
EMBED_DIM = 768   
USE_CHECKPOINTING = True 

free_memory()

encoder = Encoder(input_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM).to(device)
attention_layer = Attention(hidden_dim=HIDDEN_DIM).to(device)
decoder = Decoder(vocab_size=tokenizer.vocab_size, embedding_dim=EMBED_DIM,
                   hidden_dim=HIDDEN_DIM, attention=attention_layer).to(device)

model = Seq2Seq(encoder, decoder, use_checkpointing=USE_CHECKPOINTING).to(device)


if n_gpus > 1:
    model = nn.DataParallel(model, device_ids=list(range(n_gpus)))
    print(f"Model replicated across {n_gpus} GPUs")
else:
    print("Training on a single GPU")

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
scaler = GradScaler("cuda")

show_gpu_mem("after model build")


Model replicated across 2 GPUs
[after model build] GPU 0: 2.40 GB allocated, 2.47 GB reserved
[after model build] GPU 1: 0.00 GB allocated, 0.00 GB reserved


In [ ]:
def train_one_epoch(model, multi_model, eng_ids, eng_mask, tel_ids, tel_mask,
                     optimizer, criterion, scaler, batch_size=16):
    model.train()
    total_loss, n_batches = 0.0, 0
    n = eng_ids.size(0)
    perm = torch.randperm(n)

    for start in range(0, n, batch_size):
        idx = perm[start:start+batch_size]

        eng_ids_batch = eng_ids[idx].to(device)
        eng_mask_batch = eng_mask[idx].to(device)
        tel_ids_batch = tel_ids[idx].to(device)
        tel_mask_batch = tel_mask[idx].to(device)

        with torch.no_grad():
            src_embedded = multi_model(
                input_ids=eng_ids_batch, attention_mask=eng_mask_batch
            ).last_hidden_state

        real_len = tel_mask_batch.sum(dim=1).max().item()
        tel_batch = tel_ids_batch[:, :real_len]

        optimizer.zero_grad()
        with autocast("cuda"):
            outputs = model(src_embedded, tel_batch)        
            targets = tel_batch[:, 1:]
            loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        n_batches += 1

        del outputs, loss, src_embedded, eng_ids_batch, eng_mask_batch, tel_ids_batch, tel_mask_batch, tel_batch
        torch.cuda.empty_cache()

    return total_loss / n_batches


In [ ]:
def estimate_epoch_time(batch_size=16, n_batches_to_time=15):
    model.train()
    perm = torch.randperm(encoded_eng["input_ids"].size(0))
    times = []
    for start in range(0, n_batches_to_time * batch_size, batch_size):
        t0 = time.time()
        idx = perm[start:start+batch_size]

        eng_ids_batch = encoded_eng["input_ids"][idx].to(device)
        eng_mask_batch = encoded_eng["attention_mask"][idx].to(device)
        tel_ids_batch = encoded_tel["input_ids"][idx].to(device)
        tel_mask_batch = encoded_tel["attention_mask"][idx].to(device)

        with torch.no_grad():
            src_embedded = multi_model(input_ids=eng_ids_batch, attention_mask=eng_mask_batch).last_hidden_state
        real_len = tel_mask_batch.sum(dim=1).max().item()
        tel_batch = tel_ids_batch[:, :real_len]

        optimizer.zero_grad()
        with autocast("cuda"):
            outputs = model(src_embedded, tel_batch)
            targets = tel_batch[:, 1:]
            loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        torch.cuda.synchronize()
        times.append(time.time() - t0)
        del outputs, loss, src_embedded, eng_ids_batch, eng_mask_batch, tel_ids_batch, tel_mask_batch, tel_batch
        torch.cuda.empty_cache()

    avg = sum(times) / len(times)
    batches_per_epoch = encoded_eng["input_ids"].size(0) // batch_size
    print(f"Avg batch time: {avg:.3f}s | batches/epoch: {batches_per_epoch} "
          f"| est. epoch time: {avg*batches_per_epoch/60:.1f} min")

estimate_epoch_time(batch_size=16)
show_gpu_mem("after timing probe")


Avg batch time: 2.451s | batches/epoch: 625 | est. epoch time: 25.5 min
[after timing probe] GPU 0: 6.31 GB allocated, 7.74 GB reserved
[after timing probe] GPU 1: 0.02 GB allocated, 0.04 GB reserved


In [ ]:
N_EPOCHS = 1
BATCH_SIZE = 16
CHECKPOINT_DIR = Path("/kaggle/working")

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    avg_loss = train_one_epoch(
        model, multi_model,
        encoded_eng["input_ids"], encoded_eng["attention_mask"],
        encoded_tel["input_ids"], encoded_tel["attention_mask"],
        optimizer, criterion, scaler, batch_size=BATCH_SIZE,
    )
    elapsed = time.time() - t0
    print(f"Epoch {epoch}/{N_EPOCHS} | avg loss: {avg_loss:.4f} | time: {elapsed/60:.1f} min")

    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state_dict, CHECKPOINT_DIR / f"seq2seq_epoch{epoch}.pt")
    show_gpu_mem(f"end of epoch {epoch}")


In [ ]:
@torch.no_grad()
def translate(sentence, max_len=40):
    core_model = model.module if isinstance(model, nn.DataParallel) else model
    core_model.eval()
    multi_model.eval()

    enc = tokenizer([sentence], padding="max_length", truncation=True,
                     max_length=Max_length, return_tensors="pt").to(device)
    src_embedded = multi_model(**enc).last_hidden_state

    encoder_outputs, (hidden, cell) = core_model.encoder(src_embedded)

    x = torch.tensor([[tokenizer.cls_token_id]], device=device) 
    generated = []
    for _ in range(max_len):
        prediction, hidden, cell = core_model.decoder(x, hidden, cell, encoder_outputs)
        next_id = prediction.argmax(dim=-1)          
        token_id = next_id.item()
        if token_id == tokenizer.sep_token_id:        
            break
        generated.append(token_id)
        x = next_id

    return tokenizer.decode(generated, skip_special_tokens=True)

print(translate(train_eng[0]))
print("Reference:", train_tel[0])


In [ ]:
torch.save(core_model.state_dict(), "model_weights.pt")

with open( "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)